# Controlling Output & Structured JSON in Google Gemini

Companion notebook to Anthropic's **Controlling Output & Structured Data** lesson, implemented with the free **Google GenAI SDK** (`gemini-3.5-flash-lite`).

### Architectural Comparison: Claude vs. Gemini
- **Claude's Approach**: Assistant Message Prefilling (`add_assistant_message("```json")`) + `stop_sequences=["```"]`.
- **Gemini's Approach**: Native Logit-Constrained JSON Mode (`response_mime_type="application/json"`) & Pydantic Schema Enforcement (`response_schema`).

> ⚠️ **Important API Note**: Gemini's API requires requests to end with a user turn. If you attempt an assistant prefill by ending with a `model` turn, the Gemini API raises `ClientError: 400 Requests ending with a model turn are not supported`. Instead, Gemini solves this natively at the engine level!

In [ ]:
# Install dependencies
%pip install google-genai pydantic python-dotenv

In [ ]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Initialize Gemini Client
from google import genai
from google.genai import types

client = genai.Client()
model = "gemini-3.5-flash-lite"

--- 
## 1. Stop Sequences in Gemini
Just like Claude, Gemini supports `stop_sequences` passed inside `types.GenerateContentConfig`.

In [ ]:
config = types.GenerateContentConfig(
    stop_sequences=["5", "3, 4"],
    temperature=0.0
)

response = client.models.generate_content(
    model=model,
    contents="Count from 1 to 10 separated by commas",
    config=config
)

print("Stop Sequence Output:")
print(repr(response.text))

--- 
## 2. Gemini's Native JSON Mode (`response_mime_type="application/json"`)
By specifying `response_mime_type="application/json"`, Gemini mathematically constrains its token generation to produce **only valid JSON**.

Zero markdown backticks (```` ```json ````), zero conversational fluff, and zero headers!

In [ ]:
config = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.0
)

response = client.models.generate_content(
    model=model,
    contents="Generate a very short event bridge rule to monitor EC2 instances as json",
    config=config
)

print("Raw Output:")
print(response.text)

--- 
## 3. Strict Schema Enforcement with Pydantic (`response_schema`)
You can enforce exact key names and data types by passing a Pydantic model directly into `response_schema`.

In [ ]:
from pydantic import BaseModel, Field

# Define strict schema
class EventBridgeRule(BaseModel):
    source: list[str] = Field(description="AWS event source, e.g. aws.ec2")
    detail_type: list[str] = Field(description="Event notification type")
    state: list[str] = Field(description="Target EC2 state filters, e.g. running, stopped")

config = types.GenerateContentConfig(
    response_mime_type="application/json",
    response_schema=EventBridgeRule,
    temperature=0.0
)

response = client.models.generate_content(
    model=model,
    contents="Generate an EventBridge rule to monitor EC2 instances state changes",
    config=config
)

print("Schema-Enforced JSON Output:")
print(response.text)

--- 
## 4. Seamless Parsing into Python Dict
With guaranteed structured output, parsing is completely deterministic.

In [ ]:
import json

data = json.loads(response.text)
print("Parsed Dictionary:", data)
print("AWS Source:", data["source"])
print("State Filters:", data["state"])

--- 
## 5. Course Exercise: Structured AWS CLI Commands (Gemini Companion)

### The Challenge:
Generate three sample AWS CLI commands in a single response with zero comments, zero explanations, and no prompt changes.

In Gemini, we can achieve this in two clean ways:
1. **Raw CLI text**: Via `system_instruction` with temperature `0.0`.
2. **Structured JSON**: Via a Pydantic `AWSCommands` schema.

In [ ]:
# Approach A: Raw Text Commands (Zero Markdown, Zero Comments)
config = types.GenerateContentConfig(
    system_instruction="Output only the raw shell commands, one per line. No markdown backticks, no comments, no explanations.",
    temperature=0.0
)

prompt = "Generate three different sample AWS CLI commands. Each should be very short."
response = client.models.generate_content(
    model=model,
    contents=prompt,
    config=config
)

print("--- Approach A: Clean Raw Shell Commands ---")
print(response.text.strip())

In [ ]:
# Approach B: Native JSON Schema (Pure Parseable List)
class AWSCommands(BaseModel):
    commands: list[str] = Field(description="List of 3 short AWS CLI commands")

config_json = types.GenerateContentConfig(
    response_mime_type="application/json",
    response_schema=AWSCommands,
    temperature=0.0
)

response_json = client.models.generate_content(
    model=model,
    contents=prompt,
    config=config_json
)

print("--- Approach B: Pydantic Schema-Enforced JSON ---")
print(response_json.text)